# Comparative Analysis of ML Algorithms for Mobile Price Range Classification

Predict mobile price range (0=Low, 1=Medium, 2=High, 3=Very High) from specs.
Models compared: Decision Tree, Random Forest, KNN, Logistic Regression, AdaBoost.

**Run:** place `train.csv` in `../data/`, then Run All. All randomness uses `random_state=42`.

In [ ]:
import os, joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42
TRAIN_PATH = '../data/train.csv'
sns.set(style='whitegrid')

## Part 1 - Data Loading
Each display tells us something: head/tail = format sanity, shape = size, dtypes = numeric vs categorical, describe = ranges/outliers, missing/duplicates = cleaning needed.

In [ ]:
df = pd.read_csv(TRAIN_PATH)
print('Shape:', df.shape)
display(df.head())
display(df.tail())
print('Columns:', list(df.columns))
print(df.dtypes)
display(df.describe())
print('Missing values per column:')
print(df.isnull().sum())
print('Duplicate rows:', int(df.duplicated().sum()))
print('price_range counts:')
print(df['price_range'].value_counts().sort_index())

## Part 2 - Exploratory Data Analysis
We plot target balance, key specs vs price, and feature correlations. Short takeaways go after each plot (only what the data shows).

In [ ]:
# 1. Distribution of price_range
plt.figure(figsize=(6,4))
sns.countplot(x='price_range', data=df)
plt.title('Distribution of price_range (0=Low..3=Very High)')
plt.savefig('../results/plots/eda_price_distribution.png', bbox_inches='tight')
plt.show()
# OBSERVE: classes look balanced (~500 each) - no class-imbalance fix needed.

# 2-6. Key specs vs price (box plots)
for col in ['ram', 'battery_power', 'int_memory', 'pc', 'fc']:
    plt.figure(figsize=(6,4))
    sns.boxplot(x='price_range', y=col, data=df)
    plt.title(f'{col} vs price_range')
    plt.savefig(f'../results/plots/eda_{col}_vs_price.png', bbox_inches='tight')
    plt.show()
# OBSERVE: ram rises clearly with price (strongest visual signal). battery_power overlaps a lot (weak signal). pc/fc rise slightly.

# 7-8. Screen size & pixel resolution vs price
df['screen_area'] = df['sc_h'] * df['sc_w']
df['pixel_area'] = df['px_height'] * df['px_width']
plt.figure(figsize=(6,4))
sns.boxplot(x='price_range', y='screen_area', data=df)
plt.title('screen_area (sc_h*sc_w) vs price_range')
plt.savefig('../results/plots/eda_screen_vs_price.png', bbox_inches='tight')
plt.show()
plt.figure(figsize=(6,4))
sns.boxplot(x='price_range', y='pixel_area', data=df)
plt.title('pixel_area (px_h*px_w) vs price_range')
plt.savefig('../results/plots/eda_pixels_vs_price.png', bbox_inches='tight')
plt.show()

# 9. 3G/4G availability vs price
pd.crosstab(df['price_range'], df['three_g']).plot(kind='bar', figsize=(6,4))
plt.title('3G support by price_range')
plt.savefig('../results/plots/eda_3g_vs_price.png', bbox_inches='tight')
plt.show()
pd.crosstab(df['price_range'], df['four_g']).plot(kind='bar', figsize=(6,4))
plt.title('4G support by price_range')
plt.savefig('../results/plots/eda_4g_vs_price.png', bbox_inches='tight')
plt.show()

# 10. Correlation heatmap
plt.figure(figsize=(12,10))
sns.heatmap(df.corr(numeric_only=True), cmap='coolwarm', center=0)
plt.title('Correlation heatmap')
plt.savefig('../results/plots/eda_correlation.png', bbox_inches='tight')
plt.show()
print(df.corr(numeric_only=True)['price_range'].sort_values(ascending=False))
# OBSERVE: ram has the highest correlation with price_range; px_height/px_width correlate with each other (multicollinearity, harmless for trees).

## Part 3 - Preprocessing
- No missing values in this dataset; duplicates removed if any. All features are numeric so no encoding needed.
- Target `price_range` is never scaled. Scaling (StandardScaler) is applied ONLY inside Pipelines for KNN/Logistic Regression to avoid data leakage.
- Stratified 80/20 train/validation split keeps class ratios equal.

In [ ]:
FEATURES = [c for c in df.columns if c != 'price_range' and c not in ('screen_area','pixel_area')]
X = df[FEATURES]
y = df['price_range']
print('Duplicates removed:', int(df.duplicated().sum()))
df_clean = df.drop_duplicates().reset_index(drop=True)
X = df_clean[FEATURES]; y = df_clean['price_range']
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print('Train:', X_train.shape, 'Valid:', X_valid.shape)

## Part 4 - Train five models
- Decision Tree: single tree, max_depth=10 controls overfitting.
- Random Forest: 100 trees (n_estimators), each on a bootstrap sample; majority vote.
- KNN: distance-based, so scaled; default k=5.
- Logistic Regression: multinomial (softmax) via lbfgs solver for 4 classes; scaled; C=1.0 regularization.
- AdaBoost: boosting fits weak trees sequentially, each focusing on prior mistakes; 100 rounds, learning_rate=1.0.

In [ ]:
models = {
    'decision_tree': DecisionTreeClassifier(max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=RANDOM_STATE),
    'random_forest': RandomForestClassifier(n_estimators=100, max_depth=15, min_samples_split=5, random_state=RANDOM_STATE, n_jobs=-1),
    'knn': Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=5))]),
    'logistic_regression': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=RANDOM_STATE))]),
    'adaboost': AdaBoostClassifier(n_estimators=100, learning_rate=1.0, random_state=RANDOM_STATE),
}
for name, m in models.items():
    scores = cross_val_score(m, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    print(f'{name}: CV accuracy {scores.mean():.4f} +/- {scores.std():.4f}')
    m.fit(X_train, y_train)

## Parts 6-7 - Cross-validation (above) + light hyperparameter tuning
GridSearchCV with small grids and cv=5; scaler stays inside the pipeline so no leakage. We keep the tuned model only if it beats the baseline on validation.

In [ ]:
param_grids = {
    'decision_tree': {'max_depth': [5, 10, 15, None], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2]},
    'random_forest': {'n_estimators': [100, 200], 'max_depth': [10, 15, None], 'min_samples_split': [2, 5]},
    'knn': {'clf__n_neighbors': [3, 5, 7, 11], 'clf__weights': ['uniform', 'distance'], 'clf__metric': ['euclidean', 'manhattan']},
    'logistic_regression': {'clf__C': [0.1, 1.0, 10.0], 'clf__solver': ['lbfgs']},
    'adaboost': {'n_estimators': [50, 100], 'learning_rate': [0.5, 1.0]},
}
final_models = {}
for name, m in models.items():
    grid = GridSearchCV(m, param_grids[name], cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    base = accuracy_score(y_valid, m.predict(X_valid))
    tuned = accuracy_score(y_valid, grid.best_estimator_.predict(X_valid))
    chosen = grid.best_estimator_ if tuned >= base else m
    final_models[name] = chosen
    print(f'{name}: baseline={base:.4f} tuned={tuned:.4f} best={grid.best_params_} -> using {"tuned" if chosen is grid.best_estimator_ else "baseline"}')

## Part 5 - Evaluation (same validation set for all models)

In [ ]:
rows = []
for name, m in final_models.items():
    p = m.predict(X_valid)
    rows.append({'Model': name,
                 'Accuracy': accuracy_score(y_valid, p),
                 'Precision': precision_score(y_valid, p, average='macro', zero_division=0),
                 'Recall': recall_score(y_valid, p, average='macro', zero_division=0),
                 'F1 Score': f1_score(y_valid, p, average='macro', zero_division=0)})
    print(f'===== {name} =====')
    print(classification_report(y_valid, p, digits=4))
comparison = pd.DataFrame(rows)
display(comparison)
comparison.to_csv('../results/model_comparison.csv', index=False)
print('Saved ../results/model_comparison.csv')

# Confusion matrices
for name, m in final_models.items():
    cm = confusion_matrix(y_valid, m.predict(X_valid), labels=[0,1,2,3])
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {name}'); plt.xlabel('Predicted'); plt.ylabel('True')
    plt.savefig(f'../results/confusion_matrices/{name}.png', bbox_inches='tight')
    plt.show()

# Metric comparison bars
for metric in ['Accuracy','Precision','Recall','F1 Score']:
    plt.figure(figsize=(8,4))
    sns.barplot(data=comparison, x='Model', y=metric)
    plt.ylim(0,1); plt.xticks(rotation=15); plt.title(f'Model comparison - {metric}')
    plt.savefig(f'../results/plots/{metric.lower().replace(" ","_")}_comparison.png', bbox_inches='tight')
    plt.show()

## Part 8 - Feature importance (Random Forest & Decision Tree)
Importance = how much the model relies on a feature, NOT causation.

In [ ]:
import numpy as np
def _importances(m):
    return m.named_steps['clf'].feature_importances_ if isinstance(m, Pipeline) else m.feature_importances_
for name in ['random_forest', 'decision_tree']:
    imp = _importances(final_models[name])
    order = np.argsort(imp)[::-1]
    plt.figure(figsize=(8,5))
    sns.barplot(x=imp[order], y=np.array(FEATURES)[order])
    plt.title(f'Feature importance - {name}')
    plt.savefig(f'../results/plots/feature_importance_{name}.png', bbox_inches='tight')
    plt.show()
    print(name, 'top-5:', [(FEATURES[i], round(float(imp[i]),4)) for i in order[:5]])

## Part 9 - Prediction system (uses the trained model, no hard-coded rules)

In [ ]:
import joblib
os.makedirs('../models', exist_ok=True)
for name, m in final_models.items():
    joblib.dump(m, f'../models/{name}.pkl')
print('Models saved with joblib.')
LABELS = {0:'Low Cost', 1:'Medium Cost', 2:'High Cost', 3:'Very High Cost'}
sample = {'battery_power':3000,'blue':1,'clock_speed':2.0,'dual_sim':1,'fc':8,'four_g':1,'int_memory':64,'m_dep':0.5,'mobile_wt':150,'n_cores':8,'pc':16,'px_height':1200,'px_width':1800,'ram':4096,'sc_h':14,'sc_w':7,'talk_time':12,'three_g':1,'touch_screen':1,'wifi':1}
rf = joblib.load('../models/random_forest.pkl')
X_new = pd.DataFrame([sample])
pred = int(rf.predict(X_new)[0])
proba = rf.predict_proba(X_new)[0]
print(f'Predicted Price Range: {LABELS[pred]} (class {pred})')
for i, p in enumerate(proba):
    print(f'{LABELS[i]}: {p*100:.1f}%')